# Gene-Based Aggregation — AA Smoking Status

**Purpose:** Replace individual SNP-level features with gene-level burden scores,
so the causal pipeline (DoubleML → stability selection → PC algorithm) operates on
genes instead of SNPs. Motivation: different SNPs can tag the same underlying gene
in different ancestries due to differing LD structure, so SNP-level comparison
across AA/EA can miss real shared biology that gene-level comparison can catch.

**Method:**
1. Pull all GRCh37 gene coordinates (chrom, start, end, gene type) in bulk via
   BioMart — one query, not per-SNP lookups (per-SNP Ensembl REST calls for
   ~141k SNPs would be impractical).
2. Map each SNP to its chromosome + position via the Illumina ExomeChip manifest
   (HumanExome-12-v1-0-B), matched on stripped probe ID.
3. Assign each SNP to a gene via local interval matching (SNP position falls
   within a gene's start–end range). SNPs outside any gene are labeled
   intergenic and excluded from burden aggregation.
4. Filter gene reference to `protein_coding` genes only (drops pseudogenes,
   lincRNA, miRNA, etc. — excluded for biological interpretability).
5. **Signed burden score:** for each SNP, compute the sign of its correlation
   with the phenotype in this cohort. Flip dosage (0↔2) for SNPs with a
   negative correlation, so all SNPs in a gene point the same direction before
   summing. This avoids cancellation when a gene contains both risk-increasing
   and risk-decreasing SNPs — a naive unsigned sum would understate or hide
   real gene-level signal.
6. Sum signed dosages per gene, per person, to produce the final gene burden
   matrix (genes × samples).

**Inputs:**
- `checkpoint7b_snp_encoded_012_relatedness_filtered.csv` (genotypes, 3036 samples)
- `checkpoint2b_metadata_relatedness_filtered.csv` (phenotype, smoking_status)
- `HumanExome-12-v1-0-B.csv` (Illumina manifest, probe → chr/position)
- BioMart GRCh37 gene export (chrom/start/end/gene type, all genes)

**Outputs:**
- `snp_to_gene_map_full.json` — 141,324 SNPs mapped; 131,483 genic, 9,841 intergenic
- `grch37_genes_clean.csv` — 57,773 genes, standard chromosomes only
- `gene_burden_matrix_signed_protein_coding.csv` — final input matrix,
  **15,309 protein-coding genes × 3,036 samples**

**Note on SNPs-per-gene:** distribution is right-skewed (median 5, max 296
SNPs/gene). Genes with very few SNPs (2,667 genes have only 1) get essentially
SNP-level signal; genes with many SNPs benefit most from the signed-burden
correction.

In [1]:
import pandas as pd
import numpy as np
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

# Final signed, protein-coding gene burden matrix
gene_burden = pd.read_csv(os.path.join(out_dir, "gene_burden_matrix_signed_protein_coding.csv"), index_col=0)
gene_names = gene_burden.index.tolist()
print("Gene burden matrix:", gene_burden.shape)

# Metadata / phenotype, aligned to genotype sample columns
meta_df = pd.read_csv(os.path.join(out_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
meta_df = meta_df.set_index("sample_id")
meta_df["smoking_status_bin"] = (meta_df["smoking_status"] == "Smoker").astype(int)

sample_cols = gene_burden.columns.tolist()
pheno = meta_df.loc[sample_cols, "smoking_status_bin"].astype(float)
print("Phenotype aligned:", pheno.shape)
print(pheno.value_counts())

# Confounders (same 12-col block used at SNP level)
X_confounders = np.load(os.path.join(out_dir, "confounders_X_relatedness_filtered.npy"))
print("Confounders:", X_confounders.shape)

Gene burden matrix: (15309, 3036)
Phenotype aligned: (3036,)
smoking_status_bin
0.0    1577
1.0    1459
Name: count, dtype: int64
Confounders: (3036, 12)


In [2]:
from sklearn.model_selection import KFold
from scipy import stats
import time

X_genes = gene_burden.T.values  # samples x genes
X_standardized = (X_genes - X_genes.mean(axis=0)) / X_genes.std(axis=0)
Y = pheno.values

def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)
    Xc = np.column_stack([np.ones(n), X_conf])

    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D

    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid

    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()

    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))
    return theta, pvals

n_repeats = 30
threshold = 0.001
n_genes = X_standardized.shape[1]
significant_counts = np.zeros(n_genes, dtype=int)

start = time.time()
for rep in range(n_repeats):
    theta_rep, pval_rep = doubleml_scan(X_standardized, Y, X_confounders, random_state=rep)
    significant_counts += (pval_rep < threshold).astype(int)
    if (rep + 1) % 5 == 0:
        print(f"Completed {rep+1}/{n_repeats}, elapsed {time.time()-start:.1f}s")

print(f"Total time: {time.time()-start:.1f}s")

stability_fraction = significant_counts / n_repeats
results_df = pd.DataFrame({
    "gene": gene_names,
    "stability_fraction": stability_fraction,
    "n_significant_repeats": significant_counts
}).sort_values("stability_fraction", ascending=False)

print("\nStability distribution:")
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {t:.0%}: {(stability_fraction >= t).sum()} genes")

results_df.to_csv(os.path.join(out_dir, "gene_doubleml_stability_smoking_AA.csv"), index=False)
print("\nSaved.")

Completed 5/30, elapsed 19.3s
Completed 10/30, elapsed 38.0s
Completed 15/30, elapsed 56.6s
Completed 20/30, elapsed 75.2s
Completed 25/30, elapsed 93.7s
Completed 30/30, elapsed 112.5s
Total time: 112.5s

Stability distribution:
  >= 50%: 632 genes
  >= 60%: 490 genes
  >= 70%: 371 genes
  >= 80%: 275 genes
  >= 90%: 190 genes
  >= 100%: 66 genes

Saved.


In [4]:
r2_matrix_arr = r2_matrix.values.copy()
np.fill_diagonal(r2_matrix_arr, 0)
pairs = np.argwhere(r2_matrix_arr > 0.99)

seen = set()
redundant_pairs = []
for i, j in pairs:
    if i < j:
        gene_i = shortlist_100["gene"].iloc[i]
        gene_j = shortlist_100["gene"].iloc[j]
        redundant_pairs.append((gene_i, gene_j, r2_matrix_arr[i, j]))

print(f"Near-duplicate gene pairs (r² > 0.99): {len(redundant_pairs)}")
for g1, g2, r2 in redundant_pairs:
    print(f"  {g1} <-> {g2}: r² = {r2:.4f}")

Near-duplicate gene pairs (r² > 0.99): 0


In [5]:
import pandas as pd
import numpy as np
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

results_df = pd.read_csv(os.path.join(out_dir, "gene_doubleml_stability_smoking_AA.csv"))
shortlist_100 = results_df[results_df["stability_fraction"] == 1.0].copy()
print("Genes at 100% stability:", len(shortlist_100))
print(shortlist_100["gene"].tolist())

# Pull their burden scores
gene_burden = pd.read_csv(os.path.join(out_dir, "gene_burden_matrix_signed_protein_coding.csv"), index_col=0)
shortlist_burden = gene_burden.loc[shortlist_100["gene"]]
print("\nShortlist burden matrix:", shortlist_burden.shape)

# Correlation check: find near-duplicate genes (r^2 > 0.99)
corr_matrix = shortlist_burden.T.corr()  # genes x genes
r2_matrix = corr_matrix ** 2

# Find pairs above threshold (excluding self-correlation on the diagonal)
np.fill_diagonal(r2_matrix.values, 0)
pairs = np.argwhere(r2_matrix.values > 0.99)

seen = set()
redundant_pairs = []
for i, j in pairs:
    if i < j:
        gene_i = shortlist_100["gene"].iloc[i]
        gene_j = shortlist_100["gene"].iloc[j]
        redundant_pairs.append((gene_i, gene_j, r2_matrix.values[i, j]))

print(f"\nNear-duplicate gene pairs (r² > 0.99): {len(redundant_pairs)}")
for g1, g2, r2 in redundant_pairs:
    print(f"  {g1} <-> {g2}: r² = {r2:.4f}")

Genes at 100% stability: 66
['F10', 'FREM2', 'ZNF805', 'DNAH1', 'STIL', 'DNAH11', 'FAM179A', 'FAM126A', 'PCDH12', 'LRP2', 'KIAA1377', 'GPR98', 'DVL1', 'ASTN1', 'LAMA5', 'FBN3', 'DNAH5', 'HYDIN', 'DCHS2', 'SIGLEC1', 'TAS1R3', 'LIMS2', 'SPEF2', 'ATG2A', 'EXO1', 'NLRP8', 'LOXHD1', 'CP', 'IL19', 'USH2A', 'BCLAF1', 'TEP1', 'TG', 'TICAM1', 'LZTFL1', 'TRIM45', 'TSC2', 'CELSR2', 'HMCN1', 'CENPF', 'RTN4', 'PLEC', 'UTP20', 'RYR1', 'CLIP1', 'FRAS1', 'DLEC1', 'CEP135', 'PET112', 'IGSF5', 'CNTRL', 'XIRP1', 'CCBL2', 'CHD6', 'C4orf22', 'RNF213', 'MDN1', 'ZNF418', 'COL6A6', 'CHPF2', 'CPAMD8', 'PLXNA2', 'OR51I1', 'FREM1', 'LAMA3', 'AXDND1']

Shortlist burden matrix: (66, 3036)


ValueError: underlying array is read-only

In [6]:
r2_matrix_arr = r2_matrix.values.copy()
np.fill_diagonal(r2_matrix_arr, 0)
pairs = np.argwhere(r2_matrix_arr > 0.99)

seen = set()
redundant_pairs = []
for i, j in pairs:
    if i < j:
        gene_i = shortlist_100["gene"].iloc[i]
        gene_j = shortlist_100["gene"].iloc[j]
        redundant_pairs.append((gene_i, gene_j, r2_matrix_arr[i, j]))

print(f"Near-duplicate gene pairs (r² > 0.99): {len(redundant_pairs)}")
for g1, g2, r2 in redundant_pairs:
    print(f"  {g1} <-> {g2}: r² = {r2:.4f}")

Near-duplicate gene pairs (r² > 0.99): 0


In [7]:
import numpy as np
import json
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

X_genes_pc = shortlist_burden.T.values  # samples x genes (3036 x 66)
Y_pc = pheno.values.reshape(-1, 1)      # samples x 1

X_pc_full = np.hstack([X_genes_pc, Y_pc])
col_names = shortlist_burden.index.tolist() + ["smoking_status"]

print("PC input shape:", X_pc_full.shape)
print("n_nodes:", len(col_names))
print("Outcome index:", col_names.index("smoking_status"))

np.save(os.path.join(out_dir, "pc_input_smoking_genes_AA.npy"), X_pc_full)
with open(os.path.join(out_dir, "pc_col_names_smoking_genes_AA.json"), "w") as f:
    json.dump(col_names, f)
print("Saved.")

PC input shape: (3036, 67)
n_nodes: 67
Outcome index: 66
Saved.
